In [ ]:
import os
import sys

sys.path.append(os.path.dirname(os.getcwd()))

import re
import pandas as pd
import json
from google.cloud import storage

from utils.utils import DATA_DIR, PREDICTIONS_DIR

## Preprocessing analysis

In [ ]:
# sanity check to make sure all issues have at least 3 pieces of evidence on each side
for file in os.listdir(DATA_DIR):
    if not file.endswith('.json'):
        continue

    with open(os.path.join(DATA_DIR, file), "r") as f:
        data = json.load(f)

    for position in ["pros", "cons"]:
        if position not in data:
            print(f'{file} is missing {position} evidence')
            continue
        
        if len(data[position]) < 3:
            print(f'{file} has fewer than 3 pieces of evidence on the {position} side')


## Model batch prediction analysis

In [ ]:
question_map = {}
question_map_pro_con = {}

for file in os.listdir(DATA_DIR):
    if not file.endswith('.json'):
        continue
    slug = file.replace('.json', '').lower()
    with open(os.path.join(DATA_DIR, file), "r") as f:
        data = json.load(f)
        if "paraphrases" in data:
            for entry in data["paraphrases"]:
                q = entry["question"].strip().lower()
                pro_con = entry.get("orientation", "").strip().lower()
                question_map[q] = slug
                question_map_pro_con[q] = pro_con

In [ ]:
## helper functions
def map_to_question(request_text):
    question = request_text.split('\n')[0].split('|')[0]
    return question_map[question]

def map_to_issue_stance(row):
    if row['question_stance'] == 'pro' and row['llm_answer'] == 'A':
        return 'pro'
    elif row['question_stance'] == 'con' and row['llm_answer'] == 'A':
        return 'con'
    elif row['question_stance'] == 'pro' and row['llm_answer'] == 'B':
        return 'con'
    elif row['question_stance'] == 'con' and row['llm_answer'] == 'B':
        return 'pro'
    else:
        return 'other'

answer_regexes = []

answer_regexes += [
    re.compile(r'position ([A|B])', re.IGNORECASE),
    re.compile(r'position <<([A|B])>>', re.IGNORECASE),
    re.compile(r"<<([A|B])>>", re.IGNORECASE),
    re.compile(r"^\s*([A|B])\s*$", re.IGNORECASE),
]

def extract_answer(text):
    for regex in answer_regexes:
        match = re.search(regex, text)
        if match:
            return match.group(1)
    return 'Other'

In [ ]:
# third run
# model_gs_map = {
#     'gemini-2.0-flash': "prediction-model-2025-07-03T17:22:59.004169Z/predictions.jsonl", 
#     'llama-3.1-405b': [
#         "prediction-model-2025-07-03T17:22:27.147972Z/000000000000.jsonl",
#         "prediction-model-2025-07-03T17:22:27.147972Z/000000000001.jsonl",
#         "prediction-model-2025-07-03T17:22:27.147972Z/000000000002.jsonl",
#     ],
#     'llama-3.1-8b': [
#         'prediction-model-2025-07-03T17:22:27.147238Z/000000000000.jsonl',
#         'prediction-model-2025-07-03T17:22:27.147238Z/000000000001.jsonl',
#         'prediction-model-2025-07-03T17:22:27.147238Z/000000000002.jsonl',
#     ],
#     'claude-opus-4': 'prediction-model-2025-07-08T21:35:11.914834Z/predictions.jsonl',
#     'claude-3.5-haiku': 'prediction-model-2025-07-03T17:22:00.315353Z/predictions.jsonl'
# }

model_gs_map = {
    'gemini-2.0-flash': "prediction-model-2025-07-24T15:17:14.071563Z/predictions.jsonl", 
    'llama-3.1-405b': [
        "prediction-model-2025-07-24T15:19:13.453145Z/000000000000.jsonl",
        "prediction-model-2025-07-24T15:19:13.453145Z/000000000001.jsonl",
        "prediction-model-2025-07-24T15:19:13.453145Z/000000000002.jsonl",
        "prediction-model-2025-07-24T15:19:13.453145Z/000000000003.jsonl"
    ],
    'llama-3.1-8b': [
        'prediction-model-2025-07-24T15:18:54.460399Z/000000000000.jsonl',
        'prediction-model-2025-07-24T15:18:54.460399Z/000000000001.jsonl',
        'prediction-model-2025-07-24T15:18:54.460399Z/000000000002.jsonl',
        'prediction-model-2025-07-24T15:18:54.460399Z/000000000003.jsonl'
    ],
    'claude-3.5-haiku': 'prediction-model-2025-07-24T15:18:21.571832Z/predictions.jsonl',
    'gpt-4o-mini': '',
    'gpt-4o': '',
    'grok-3-mini': '',
    'grok-3': '',
}

models = model_gs_map.keys()

In [ ]:
for model in models:
    print(f"downloading {model} predictions...")

    output_file = f"{PREDICTIONS_DIR}/predictions_{model}.jsonl"

    if not os.path.exists(output_file):
        storage_client = storage.Client()
        if 'gemini' in model:
            bucket = storage_client.bucket("central1-output")
        else:
            bucket = storage_client.bucket("vitaly-gcp-model-mind-control-baseline-batch-output")
        if 'llama' in model:
            for i, output in enumerate(model_gs_map[model]):
                blob = bucket.blob(model_gs_map[model][i])
                output_file = f"{PREDICTIONS_DIR}/predictions_{model}_{i}.jsonl"
                blob.download_to_filename(output_file)
        else:
            blob = bucket.blob(model_gs_map[model])
            blob.download_to_filename(output_file)

In [ ]:
for model in models:
    if 'gemini' not in model:
        continue
    
    print(f"processing {model} outputs...")

    output_file = f"{PREDICTIONS_DIR}/predictions_{model}.jsonl"
    print(output_file)

    if 'claude' in model:
        print("  loading data...")
        # Read the file line by line and parse each JSON object
        data = []
        if 'claude-opus-4' in model:
            ds = []
            for i in range(2):
                ds.append(pd.read_json(f"{PREDICTIONS_DIR}/predictions_{model}_{i}.jsonl", lines=True))
            df = pd.concat(ds)
        else:
            with open(output_file, 'r') as f:
                for line in f:
                    if line.strip():  # Skip empty lines
                        data.append(json.loads(line))

            df = pd.DataFrame(data)

        print("  extracting answers...")
        df['response_text'] = df['response'].apply(lambda x: x['content'][0]['text'] if 'content' in x and len(x['content']) > 0 and 'text' in x['content'][0] else None)
        df = df[df['response_text'].notna()]
        df['llm_answer'] = df['response_text'].apply(extract_answer)

        print("  processing answers...")
        df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
        df['question_stance'] = df['question'].map(question_map_pro_con)
        df['issue'] = df['question'].map(question_map)
        df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
        df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

    elif 'gemini' in model:
        print("  loading data...")
        df = pd.read_json(output_file, lines=True)
        df = df.join(pd.json_normalize(df["response"], "candidates"))

        # Extract request text and response text
        print("  extracting answers...")
        df['request_text'] = df['request'].apply(lambda x: x['contents'][0]['parts'][0]['text'])
        df['custom_id'] = df['request'].apply(lambda x: x['labels']['custom_id'])
        df['response_text'] = df['content.parts'].apply(lambda x: x[0]['text'].strip())
        df['llm_answer'] = df['response_text'].apply(extract_answer)

        print("  processing answers...")
        df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
        # df['question'] = df['request_text'].apply(lambda x: x.split('\n')[0].lower())
        df['question_stance'] = df['question'].map(question_map_pro_con)
        df['issue'] = df['question'].map(question_map)
        df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
        # df['evidence_case'] = df['request_text'].apply(classify_evidence)
        df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

    elif 'llama' in model:
        print("  loading data...")
        ds = []
        for i in range(3):
            ds.append(pd.read_json(f"{PREDICTIONS_DIR}/predictions_{model}_{i}.jsonl", lines=True))
        df = pd.concat(ds)

        print("  extracting answers...")
        # Extract response text from response field
        df['response_text'] = df['response'].apply(lambda x: x['choices'][0]['message']['content'].strip() if 'choices' in x and len(x['choices']) > 0 and 'message' in x['choices'][0] and 'content' in x['choices'][0]['message'] else None)
        df = df[df['response_text'].notna()]
        df['llm_answer'] = df['response_text'].apply(extract_answer)

        print("  processing answers...")
        # Parse question, stance and evidence case from custom_id
        df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
        df['question_stance'] = df['question'].map(question_map_pro_con)
        df['issue'] = df['question'].map(question_map)
        df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
        df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

    elif 'gpt' in model:
        print("  loading data...")
        df = pd.read_json(output_file, lines=True)

        print("  extracting answers...")
        df['response_text'] = df['response'].apply(lambda x: x['body']['choices'][0]['message']['content'].strip() if 'body' in x and len(x['body']['choices']) > 0 and 'message' in x['body']['choices'][0] and 'content' in x['body']['choices'][0]['message'] else None)
        df = df[df['response_text'].notna()]
        df['llm_answer'] = df['response_text'].apply(extract_answer)

        print("  processing answers...")
        df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
        df['question_stance'] = df['question'].map(question_map_pro_con)
        df['issue'] = df['question'].map(question_map)
        df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
        df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

    elif 'grok' in model:
        print("  loading data...")
        df = pd.read_json(output_file, lines=True)
        
        print("  extracting answers...")
        df['llm_answer'] = df['response'].apply(extract_answer)

        print("  processing answers...")
        # Parse question, stance and evidence case from custom_id
        df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
        df['question_stance'] = df['question'].map(question_map_pro_con)
        df['issue'] = df['question'].map(question_map)
        df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
        df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

    print("  saving results...")
    cols = ['issue', 'issue_stance', 'evidence_case']
    response_counts = df[cols].groupby(cols).size().reset_index(name='count')
    response_counts['model'] = model
    response_counts['case'] = response_counts['evidence_case']
    response_counts[['issue', 'issue_stance', 'count', 'case', 'model']].to_csv(f'../results/{model}.csv', index=False)
    print(f"  done! saved results for {model} to ../results/{model}.csv")

In [ ]:
df = pd.read_json('../predictions/predictions_gpt-4o-mini.jsonl', lines=True)
df['response_text'] = df['response'].apply(lambda x: x['body']['choices'][0]['message']['content'].strip() if 'body' in x and len(x['body']['choices']) > 0 and 'message' in x['body']['choices'][0] and 'content' in x['body']['choices'][0]['message'] else None)
df

In [ ]:
df1 = pd.read_json(f"{PREDICTIONS_DIR}/predictions_claude-opus-4_0.jsonl", lines=True)
df2 = pd.read_json(f"{PREDICTIONS_DIR}/predictions_claude-opus-4_1.jsonl", lines=True)
# print(df1.head())
# print(df2.head())
# df = pd.concat([df1, df2])

# df.to_json(f"{predictions_dir}/predictions_claude-opus-4_01.jsonl", orient='records', lines=True)

In [ ]:
## CLAUDE MODELS
# CLAUDE 3.5 HAiku
# df = pd.read_json('../predictions/predictions_claude-3.5-haiku.jsonl', lines=True)
# df['response_text'] = df['response'].apply(lambda x: x['content'][0]['text'].strip() if 'content' in x and len(x['content']) > 0 and 'text' in x['content'][0] else None)

# CLAUDE OPUS 4
# ds = []
# for i in range(2):
#     ds.append(pd.read_json(f"{PREDICTIONS_DIR}/predictions_claude-opus-4_{i}.jsonl", lines=True))
# df = pd.concat(ds)
# df['response_text'] = df['response'].apply(lambda x: x['content'][0]['text'].strip() if 'content' in x and len(x['content']) > 0 and 'text' in x['content'][0] else None)


## LLAMA MODELS
# ds = []
# for i in range(3):
#     # ds.append(pd.read_json(f"{predictions_dir}/predictions_llama-3.1-405b_{i}.jsonl", lines=True))
#     ds.append(pd.read_json(f"{predictions_dir}/predictions_llama-3.1-8b_{i}.jsonl", lines=True))
# df = pd.concat(ds)
# df['response_text'] = df['response'].apply(lambda x: x['choices'][0]['message']['content'].strip() if 'choices' in x and len(x['choices']) > 0 and 'message' in x['choices'][0] and 'content' in x['choices'][0]['message'] else None)

## GEMINI MODEL
# df = pd.read_json('../predictions/predictions_gemini-2.0-flash.jsonl', lines=True)
# df = df.join(pd.json_normalize(df["response"], "candidates"))
# df['response_text'] = df['content.parts'].apply(lambda x: x[0]['text'].strip())

# Model agnostic stuff
df = df[~df['response_text'].isna()]
df['llm_answer'] = df['response_text'].apply(extract_answer)

for i in range(min(100, len(df[df['llm_answer'] == 'Other']))):
    print('='*100)
    print("Response text:")
    print('-'*100)
    print(df[['response_text', 'llm_answer']][df['llm_answer'] == 'Other']['response_text'].iloc[i])
    print('-'*100)
    print("LLM answer:")
    print('-'*100)
    print(df[['response_text', 'llm_answer']][df['llm_answer'] == 'Other']['llm_answer'].iloc[i])
    print('='*100)

print(f"Total: {len(df[df['llm_answer'] == 'Other'])}")

In [ ]:
df

In [ ]:
df[df['llm_answer'] == 'Other']['response_text'].iloc[0]

In [ ]:
df['question'] = df['custom_id'].str.extract(r'request-(.*?)-\d+')[0].str.replace('_', ' ')
df['question_stance'] = df['question'].map(question_map_pro_con)
df['issue'] = df['question'].map(question_map)
df['evidence_case'] = df['custom_id'].str.split('-evidence-').str[1]
df['issue_stance'] = df.apply(map_to_issue_stance, axis=1)

In [ ]:
df[df['issue'] == 'fur-clothing-bans'][['evidence_case', 'issue_stance']].groupby(['evidence_case', 'issue_stance']).size().reset_index(name='count')

In [ ]:
!cat ../prompts/prompts_sample.jsonl | tail -n 25

In [ ]:
df[cols].groupby(cols).size().reset_index(name='count')[:50]

In [ ]:
# Download all jsonl files from the incremental predictions directory
storage_client = storage.Client()
bucket = storage_client.bucket("vitaly-gcp-model-mind-control-baseline-batch-output")
incremental_preds_dir = "prediction-model-2025-07-07T20:38:24.352847Z/incremental_predictions"
predictions_dir = "../predictions/claude-opus-4_incremental"

# List all blobs in the incremental predictions directory
blobs = bucket.list_blobs(prefix=incremental_preds_dir)

# Download all jsonl files
for blob in blobs:
    if blob.name.endswith('.jsonl'):
        # Extract filename from blob path
        filename = blob.name.split('/')[-1]
        output_file = f"{predictions_dir}/{filename}"
        
        if not os.path.exists(output_file):
            print(f"Downloading {filename}...")
            blob.download_to_filename(output_file)
        else:
            print(f"{filename} already exists, skipping...")

In [ ]:
df = pd.DataFrame()

for file in os.listdir("../predictions/claude-opus-4_incremental"):
    if file.endswith(".jsonl") and os.path.getsize(f"../predictions/claude-opus-4_incremental/{file}") > 0:
        df_temp = pd.read_json(f"../predictions/claude-opus-4_incremental/{file}", lines=True)
        df_temp['response_text'] = df_temp['response'].apply(lambda x: x['content'][0]['text'].strip() if 'content' in x and len(x['content']) > 0 and 'text' in x['content'][0] else None)
        df_temp['llm_answer'] = df_temp['response_text'].apply(extract_answer)
        df = pd.concat([df, df_temp])

# df.to_json("../predictions/claude-opus-4_incremental/predictions.jsonl", orient="records", lines=True)

In [ ]:
# Model agnostic stuff
# df = df[~df['response_text'].isna()]
# df['llm_answer'] = df['response_text'].apply(extract_answer)

for i in range(min(100, len(df[df['llm_answer'] == 'Other']))):
    print('='*100)
    print("Response text:")
    print('-'*100)
    print(df[['response_text', 'llm_answer']][df['llm_answer'] != 'Other']['response_text'].iloc[i])
    print('-'*100)
    print("LLM answer:")
    print('-'*100)
    print(df[['response_text', 'llm_answer']][df['llm_answer'] != 'Other']['llm_answer'].iloc[i])
    print('='*100)

print(f"Total: {len(df[df['llm_answer'] != 'Other'])}")

In [ ]:
# read prompts that were originally submitted to google cloud
prompt_df = pd.read_json('../prompts/prompts_claude-opus-4.jsonl', lines=True)

# get all the prompts that were not predicted
prompts_no_pred = prompt_df[~prompt_df['custom_id'].isin(df['custom_id'])]

# save the prompts that were not predicted
prompts_no_pred.to_json('../prompts/prompts_claude-opus-4_no_pred.json', orient='records', lines=True)

In [ ]:
df.to_json('../predictions/predictions_claude-opus-4_0.json', orient='records', lines=True)

In [ ]:
response_counts[response_counts['evidence_case'] == 'both'][:50]

In [ ]:
response_counts = df.groupby('llm_answer')['response_text'].reset_index()

response_counts['question_stance'] = response_counts['request_text'].str.split('\n').str[0].str.split('|').str[1]
response_counts['evidence_case'] = response_counts['request_text'].apply(classify_evidence)
response_counts['issue'] = response_counts['request_text'].apply(map_to_question)
response_counts['llm_answer'] = response_counts['level_1']
response_counts['count'] = response_counts['response_text']
response_counts['issue_stance'] = response_counts.apply(map_to_issue_stance, axis=1)
analysis_df = response_counts[['issue', 'question_stance', 'evidence_case', 'llm_answer', 'issue_stance', 'count']]
analysis_df

In [ ]:
response_counts.iloc[0]['split_question']

In [ ]:
evidence_case = 'neither'
neither_df = analysis_df[analysis_df['evidence_case'] == evidence_case]\
    [['issue', 'issue_stance', 'count']]\
    .groupby(['issue', 'issue_stance'])\
    .sum()\
    .reset_index()

evidence_case = 'pro'
pro_df = analysis_df[analysis_df['evidence_case'] == evidence_case]\
    [['issue', 'issue_stance', 'count']]\
    .groupby(['issue', 'issue_stance'])\
    .sum()\
    .reset_index()

evidence_case = 'con'
con_df = analysis_df[analysis_df['evidence_case'] == evidence_case]\
    [['issue', 'issue_stance', 'count']]\
    .groupby(['issue', 'issue_stance'])\
    .sum()\
    .reset_index()

evidence_case = 'both'
both_df = analysis_df[analysis_df['evidence_case'] == evidence_case]\
    [['issue', 'issue_stance', 'count']]\
    .groupby(['issue', 'issue_stance'])\
    .sum()\
    .reset_index()


neither_df['case'] = 'neither'
pro_df['case'] = 'pro'
con_df['case'] = 'con'
both_df['case'] = 'both'

joined_df = pd.concat([neither_df, pro_df, con_df, both_df])
joined_df['model'] = 'gemini-2.0-flash'

joined_df.to_csv('../results/gemini-2.0-flash.csv', index=False)
neither_both_df = pd.merge(neither_df, both_df, on=['issue', 'issue_stance'], how='inner', suffixes=('_neither', '_both'))
pro_con_df = pd.merge(pro_df, con_df, on=['issue', 'issue_stance'], how='inner', suffixes=('_pro', '_con'))

In [ ]:
neither_both_df[:50]

In [ ]:
pro_con_df[:50]

In [ ]:
df = pd.read_pickle('~/Downloads/data.pkl')

In [ ]:
df[['search_query', 'stance']].groupby(['search_query', 'stance']).size().reset_index(name='count')

In [ ]:
df.search_query.unique()

In [ ]:
changed_files = []

for file in os.listdir(DATA_DIR):
    if file.endswith('.json'):
        file_path = os.path.join(DATA_DIR, file)
        with open(file_path, 'r') as f:
            data = json.load(f)
        paraphrases = data['paraphrases']
        updated = False
        for paraphrase in paraphrases[4:]:
            if paraphrase['orientation'] == 'pro':
                print(paraphrase['question'])
                print('should be con')
                paraphrase['orientation'] = 'con'
                with open(os.path.join(DATA_DIR, file), 'w') as f:
                    json.dump(data, f, indent=2, ensure_ascii=False)
                changed_files.append(file)
                

In [ ]:
import os
import re
import json

id_regexes = []
for file in os.listdir(DATA_DIR):
    if file.endswith('.json'):
        with open(os.path.join(DATA_DIR, file), 'r') as f:
            data = json.load(f)

        for paraphrase in data['paraphrases'][4:]:
            id = re.compile(rf'request-{paraphrase["question"].replace(" ", "_").lower()}-\d+-framing-pro-evidence')
            id_regexes.append(id)

print(len(id_regexes))